<a href="https://colab.research.google.com/github/meskeremg/UT-Austin-AIML-Post-Graduate-Program/blob/main/Full_Code_NLP_RAG_Project_Meskerem_Goshime_January_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NLP RAG Project - Medical Assistant
###Meskerem Goshime, January 26, 2026

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q


In [ ]:
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#importing libraries
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from huggingface_hub import hf_hub_download
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA


## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
#importing the LLM model
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

# Download the model file from HuggingFace Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

In [ ]:
#instantiate the model

llm = LlamaCpp(
    model_path=model_path,
    #temperature=0.01,
    #top_p=0.95,
    n_ctx=2300,
    n_batch=512,
    n_gpu_layers=38,
    #verbose=True,
)

#### Response

In [ ]:
#define the response function
def response(query,max_tokens=2000,temperature=0,top_p=0.95,top_k=20):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query1 = "What is the protocol for managing sepsis in a critical care unit?"

In [ ]:
response(query1)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"


In [ ]:
response(query2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

In [ ]:
response(query3)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


In [ ]:
response(query4)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"


In [ ]:
response(query5)

Observations:
- The responses with LLM were decent.
- However this will not work in a medical setting as the model is not using authoritative sources.

## Question Answering using LLM with Prompt Engineering

Note:
- In the question answering with LLM section, some of the responses were very long. Therefore, I will reduce the max_tokens size to 500 from 2000 in this section.
- I will also include instruction to respond conscisely in the system prompt.
- I will also include instruction to respond as I don't have enough information when appropriate.

In [ ]:
# defining the system prompt
system_prompt = """
[INST]<<SYS>> Respond to the user question based on the user prompt.
Please be conscise when responding.
If you don't have enough information on the topic,
please respond by I don't have enough information on the topic.<</SYS>>[/INST]
"""

In [ ]:
#defining the response function
def response_prompt(query,max_tokens=500,temperature=0,top_p=0.95,top_k=20):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
#combining the system prompt with the query
prompt1 = system_prompt+"\n"+query1

In [ ]:
response_prompt(prompt1)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt2 = system_prompt+"\n"+query2

In [ ]:
response_prompt(prompt2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt3 = system_prompt+"\n"+query3

In [ ]:
response_prompt(prompt3)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt4 = system_prompt+"\n"+query4

In [ ]:
response_prompt(prompt4)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt5 = system_prompt+"\n"+query5

In [ ]:
response_prompt(prompt5)

Observations:
- In spite of including instructions to respond concisely, the responses were cut off mid-sentence on most of the responses.

### Fine Tuining

In [ ]:
# Original settings max_tokens=500,temperature=0,top_p=0.95,top_k=20
response_prompt(prompt5, max_tokens=2000)

Observations:
- Max_tokens of 2000 provided a very long response. Inspite of the length, the response still cut of mid-sentence.
- In my RAG model, I plan to use max_tokens value of 500, but be specific in the system message about the number of words the model should use .

In [ ]:
# Original settings max_tokens=500,temperature=0,top_p=0.95,top_k=20
# testing temperature value 0.5
response_prompt(prompt5, temperature=0.5)

Observations:
- There was not a significant difference in the reponse with temperature value of 0 and 0.5.
- However, I would like to stick to a lower tempreture value to prevent hallucinations.

In [ ]:
# Original settings max_tokens=500,temperature=0,top_p=0.95,top_k=20
# testing top_p value 0.5
response_prompt(prompt5, top_p=0.5)

Observation:
- There was not a noticable difference betweeen the values of top_p 0.5 and the original 0.95.

In [ ]:
# Original settings max_tokens=500,temperature=0,top_p=0.95,top_k=20
#testing top_k value 50
response_prompt(prompt5, top_k=50)

Observation:
- There was not a noticable difference in the response between the top_k values of 50 and 20.

## Data Preparation for RAG

### Loading the Data

In [ ]:
# connecting to google colab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# location of the document pdf file
pdf_path = '/content/drive/MyDrive/RAG/medical_diagnosis_manual.pdf'

In [ ]:
# Load the PDF
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader(pdf_path)
documents = loader.load()   # 1 Document per page

print(f"Total pages loaded: {len(documents)}")

In [ ]:
# removing white spaces
for d in documents:
    assert d.page_content.strip()

### Data Overview

#### Checking the first 5 pages

In [ ]:
# printing the first 5 pages
for i in range(0,5):
  print(documents[i],"/n", "/n")

#### Checking the number of pages

In [ ]:
# Number of pages in the pdf
len(documents)

### Data Chunking

In [ ]:
# Chunk the PDF
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

document_chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(document_chunks)}")

In [ ]:
# Number of chunks
len(document_chunks)

In [ ]:
# Metadata preserved
document_chunks[0].metadata

### Embedding

In [ ]:
# Embedding model

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# embedding the first chunk
embedding1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding1

In [ ]:
# length of the embedding
len(embedding1)

### Vector Database

In [ ]:
# creating the directory to hold the vectorstore database
import json,os
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [ ]:
# Creatomg the vectorstore database

from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embedding_model,
    persist_directory=out_dir
)



### Retriever

In [ ]:
# retrieve most similar chunks
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={"k": 3})

In [ ]:
# testing the retriever
retriever.get_relevant_documents("hypertension")

### System and User Prompt Template

In [ ]:
# system message for RAG
qna_system_message = """
You are an AI medical assistant who reviews the context given from the
Merck Medical Manual and answer questions.
The context will begin with the token: ###Context.
The context contains references to specific portions of the medical manual
relevant to the user query.
User questions will begin with the token: ###Question.
Please answer only using the context provided in the input.
Start your response by "According to the Merck Medical Manual".
If the answer is not found in the context, respond "I don't know".
Please respond in about 500 words or less.
"""

In [ ]:
# user message template for RAG
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

Note:
- My experimentaiton revealed that responses are too long sometimes and they cut of mid sentence when the max_tokens value is too small.
- Therefore, to solve issues of cut-off response and too long response, I instructed the model to respond in about 500 words, which is a bit smaller than the max_tokens value of 600 I used in the below generate_rag_response function.
- The max_tokens value and the system message can be adjusted based on the users' needs.

### Response Function

In [ ]:
# The RAG response function
def generate_rag_response(user_input,k=3,max_tokens=600,temperature=0.01,top_p=0.95,top_k=20):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = '[INST]' + '\n' + qna_system_message + '\n' + user_message + '\n' + '[/INST]'

    # Generate the response
    #try:
    response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
    response = "Query: \n" + user_input + "\n \n" + "Response: \n" + response
    #except Exception as e:
    #    response = f'Sorry, I encountered the following error: \n {e}'

    return response

In [ ]:
# Testing the RAG function
rag_response0 = generate_rag_response("hypertension")
rag_response0

Observations:
- Here, I only gave a one word query. It started by defining the term followed by causes, diagnosis, treatment etc, which seems reasonable.

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
generate_rag_response(query1)

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
generate_rag_response(query2)

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
generate_rag_response(query3)

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
generate_rag_response(query4)

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
generate_rag_response(query5)

### Fine-tuning

In [ ]:
# Original settings in the rag function k=3,max_tokens=600,temperature=0.01,top_p=0.95,top_k=20
# testing k=5
generate_rag_response(query5,k=5)

Observation:
- Interestingly, k value of 5 resulted in a concise response than 3. However, I think k value of 3 is sufficient to provide a thorough response.

In [ ]:
# Original settings in the rag function k=3,max_tokens=600,temperature=0.01,top_p=0.95,top_k=20
# testing max_tokens=256
generate_rag_response(query5, max_tokens=256)

- The response cutts off mid-sentence when the max tokens size was changed from 600 to 256.
- However, with the original settings of max_tokens of 600 and instructions to limit response to 500 words or less, I have noticed that resposes cut-off mid sentence occasionaly.

In [ ]:
# Original settings in the rag function k=3,max_tokens=600,temperature=0.01,top_p=0.95,top_k=20
# testing tempreture=0.5
generate_rag_response(query5, temperature=0.5)

Observation:
- I noticed that content was a little different with tempreture value of 0.5 (example: "Additional treatment may involve low-dose UFH, LMWH, warfarin, newer anticoagulants such as fondaparinux, compression devices or stockings ...")
- I would want to stick to temperature value of 0.01 to ensure there are not any hallucinations.

In [ ]:
# Original settings in the rag function k=3,max_tokens=600,temperature=0.01,top_p=0.95,top_k=20
# testing top_k=50
generate_rag_response(query5, top_k=50)

Observation:
- The content of the response was mostly similar with top_k value of 50 compared to the original top_k value of 20.
- For this rag project, we want the reponse to focus on the context than the LLMs general knowledge. Therefore, I want to keep the top_k value at 20.


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
# groundedness rating instructions
groundedness_rater_system_message = groundedness_rater_system_message = """
You will rate AI-generated answers to user questions. Each input includes a question, the context used by the AI,
and the AI-generated answer, labeled as ###Question, ###Context, and ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 – Not followed at all
2 – Followed to a limited extent
3 – Followed to a good extent
4 – Mostly followed
5 – Completely followed

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1 Give explanation if the answer adheres to the metric
2 Evaluate the extent to which the metric is followed
3 Use the previous information to rate the answer using the evaluaton criteria and assign a score between 1 and 5
"""

In [ ]:
# relevance rating instructions
relevance_rater_system_message = """
You will evaluate AI-generated answers to user questions. Each input includes a question, the context used by the AI,
and the AI’s answer, labeled as ###Question, ###Context, and ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 Not followed at all
2 Followed to a limited extent
3 Followed to a good extent
4 Mostly followed
5 Completely followed

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context.

Instructions:

1 Give explanation if the context adheres to the metric considering the question as the input
2 Evaluate the extent to which the metric is followed
3 Use the previous information to rate the context using the evaluaton criteria and assign a score between 1 and 5
"""

In [ ]:
# user message
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
# evaluation function
def generate_ground_relevance_response(user_input,k=2,max_tokens=5000,temperature=0,top_p=0.95,top_k=20):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response
    #print(answer)

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    #return response_1['choices'][0]['text'],response_2['choices'][0]['text']
    return response_1, response_2

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
# evaluating the response for query 1
ground1,rel1 = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=5000)


In [ ]:
ground1

In [ ]:
groundedness1 = 5

In [ ]:
rel1

In [ ]:
relevance1 = 4

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
# evaluating the response for query 2
ground2,rel2 = generate_ground_relevance_response(user_input=query2,max_tokens=5000)

In [ ]:
ground2

In [ ]:
groundedness2 = 3

In [ ]:
rel2

In [ ]:
relevance2 = 5

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
# evaluating the response for query 3
ground3,rel3 = generate_ground_relevance_response(user_input=query3, k=1, max_tokens=5000)

In [ ]:
ground3

In [ ]:
groundedness3 = 3

In [ ]:
rel3

In [ ]:
relevance3 = 4

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
# evaluating the response for query 4
ground4,rel4 = generate_ground_relevance_response(user_input=query4,max_tokens=5000)

In [ ]:
ground4

In [ ]:
groundedness4 = 5

In [ ]:
rel4

In [ ]:
relevance4 = 4

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
# evaluating the response for query 5
ground5,rel5 = generate_ground_relevance_response(user_input=query5,max_tokens=1000)

In [ ]:
ground5

Groundedness = unclear
The model seemed to miss detecting the context provided.

In [ ]:
rel5

The model confused the context as "The context provided in the user question is that a person has fractured their leg during a hiking trip."
Although it assigned a score of 5 for Relevance, the evaluation was based on the wrong context.


In [ ]:
# putting all ratings in a dataframe

import pandas as pd
GroundRel = {'query1':[groundedness1, relevance1], 'query2':[groundedness2, relevance2] , 'query3': [groundedness3, relevance3], 'query4': [groundedness4, relevance4]}

evaluation = pd.DataFrame(GroundRel, index=["groundedness", "relevance"])
evaluation




In [ ]:
# calculating average ratings

from re import A
Avg_groundedness = evaluation.loc['groundedness'].mean()
Avg_relevance = evaluation.loc['relevance'].mean()

print("Average Groundededness Score: ", Avg_groundedness)
print("Average Relevance Score: ", Avg_relevance)

## Actionable Insights and Business Recommendations


###Business Problem:
- Healthcare professionals deal with vast volumes of medical data.
- They work in a fast paced environment with  a need for improved efficiency and patient outcome.


###Solution approach/methodology
- Develop a RAG-Based AI solution with the following goals:
- Quick access to comprehensive, reliable and up-to-date medical knowledge
- Streamline decision making
- Improve diagnostics and patient outcomes
- Standardize care practices
- Enhance efficiency


###Project Phases

####1 - Question answering using LLM
 - LLM model used was TheBloke/Mistral-7B-Instruct-v0.2-GGUF
 - The responses with LLM were decent.
 - However this will not work in a medical setting as the model is not using authoritative sources.
 - The model gave very long responses at times and responses were sometimes cut off mid-sentence.


####2 - Question answering using LLM with prompt engineering
 - used TheBloke/Mistral-7B-Instruct-v0.2-GGUF LLM model, the same model as in prooject phase 1.
 - Setting used with the LLM model: max_tokens=500,temperature=0,top_p=0.95,top_k=20
 - The system message instructed the model to respond concisely and to respond saying it doesn't have enough information when appropriate.


####3- Question answering using RAG
 - Settings used: k=3 (top 3 similar document chunks used), max_tokens=600,temperature=0.01 (very low chance of hallucinations), top_p=0.95,top_k=20
 - The Merck Medical Manual was imported and split into chunks. A chunk size of 500 with chunk overlap of 20 was used.
 - The document chunks were then embedded using the sentence-transformers/all-MiniLM-L6-v2 model.
 - Chroma database was used to store the document embeddings.
 - TheBloke/Mistral-7B-Instruct-v0.2-GGUF model was then used to generate the RAG responses. The model was fed the user query, the retrieved context and a system message to generate responses.
 - The model is instructed to use the context to respond to the user query. It is also instructed to respond in 500 words or less and to respond with "I don't know" when appropriate.
 4- Model Performance evaluation
- The same LLM model was used to evaluate the groundedness and relevance of the RAG responses generated. The evaluation function included detailed instructions on the evaluation criteria for groundedness of the responses and the relevance of the responses.
- Although it would have been better to use a different LLM model to do the evaluations, I was not able to do that because of limitation of computation resources.
- The evaluation scores were the below:
   - Average Groundedness Score:  4.0
   - Average Relevance Score:  4.25


###Business Recommendation:
 - The RAG system is designed to provide responses to users' queries using an authoritative source, the Merck Medical Manual.
 - It is designed to avoid hallucinations and respond using the related text retrieved from the manual.
 - This system runs slow on google colab environment. Implementation in a production environment requires robust computing resources.
 - I would recommend continuing to evaluate the responses, preferably using a different LLM model.
 - I would also recommend using additional authoritative resources for the responses to be more comprehensive.

<font size=6 color='blue'>Power Ahead</font>
___